# Stage 06a — Diagnose the 2022 validation candidate

This diagnostic consumes an existing solved network; it does not run the full-year solve. It reports dispatch, potential, curtailment, capacity factors, and monthly carrier totals. Reservoir/cascade physics and threshold changes are intentionally out of scope.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.artifacts import run_preflight, format_preflight_report
from src.phase06b import aggregate_dispatch_by_carrier
import pypsa
result_dir = ROOT / 'results/phase_06a'
result_dir.mkdir(parents=True, exist_ok=True)
candidate = ROOT / 'results/phase_06/network_solved_2022.nc'
if not candidate.exists(): candidate = ROOT / 'data/processed/networks/network_base_filled.nc'
network = pypsa.Network(candidate)
summary = aggregate_dispatch_by_carrier(network)
summary.to_csv(result_dir / 'dispatch_by_carrier.csv', index=False)
monthly = network.generators_t.p.groupby(network.snapshots.month).sum().T
monthly.to_csv(result_dir / 'dispatch_by_generator_month.csv')
diagnostic = {'candidate': str(candidate.relative_to(ROOT)), 'snapshots': len(network.snapshots), 'status': 'diagnostic_only', 'reservoir_refinement': 'deferred', 'thresholds_changed': False}
(result_dir / 'diagnostic_metadata.json').write_text(json.dumps(diagnostic, indent=2), encoding='utf-8')
display(summary)

Cannot find header.dxf (GDAL_DATA is not defined)


C:\Users\fpenaherrera_vaca\.conda\envs\pypsa-earth-ec\Lib\site-packages\xarray\backends\plugins.py:110: RuntimeWarning:

Engine 'cfgrib' loading failed:
Cannot find the ecCodes library



INFO:pypsa.io:Imported network network_solved_2022.nc has buses, carriers, generators, lines, links, loads, transformers


,carrier,dispatch_gwh,potential_gwh,curtailment_gwh,capacity_factor
0,biogas,30.528572,7.288320e+01,4.235463e+01,0.418870
1,biomass,70.229469,1.956800e+02,1.254505e+02,0.055558
2,hydro,23919.364646,2.980074e+04,5.881380e+03,0.526702
3,interconnection_colombia,465.830000,4.380000e+03,3.914170e+03,0.106354
4,load_shedding,0.000000,9.723600e+06,9.723600e+06,0.000000
5,oil,3229.487738,1.806207e+04,1.483258e+04,0.178799
6,onwind,3.865716,4.925154e+00,1.059438e+00,0.006638
7,solar,33.883858,3.388386e+01,0.000000e+00,0.158007


## Interpretation

Use this output to separate temporal/profile curtailment from network effects. The accounting bridge remains explicit and unresolved residuals are not modeled as losses.